# 面试问题：Reflection / Evaluator-Optimizer Loop 怎样实现，为什么有时越改越差？

**一句话回答**：把生成、评估、修订分成独立角色；Evaluator 依据固定 rubric 输出结构化缺陷和证据，Optimizer 只处理可执行缺陷。循环保留 best-so-far，并在通过门槛、增益不足、草稿重复、评价不稳定或预算耗尽时停止。自评模型可能共享盲点，因此必须用确定性检查和人工集校准。

本 Notebook 用受控报告任务实现 rubric、critique、修订、最佳版本回滚、循环检测、预算和 evaluator 偏差探针。

In [ ]:
from dataclasses import dataclass
import copy,hashlib,json,math
import numpy as np

SEED123=12301; REQUIRED123=("conclusion","evidence","limitation")
assert len(REQUIRED123)==3
assert SEED123==12301
assert hashlib.sha256(b"draft-1").hexdigest()!=hashlib.sha256(b"draft-2").hexdigest()

## 1. Rubric 必须在看候选前冻结

每个维度定义 observable criteria、分数范围、pass threshold 与 critical failure。若看完答案才改变标准，循环会追逐 evaluator 偏好。安全/事实等硬失败不能被文风高分抵消。

In [ ]:
RUBRIC123={"conclusion":{"weight":2,"critical":False},"evidence":{"weight":3,"critical":True},"limitation":{"weight":1,"critical":False}}
def rubric_score123(draft):
    present={k:int(bool(draft.get(k))) for k in RUBRIC123}; score=sum(RUBRIC123[k]["weight"]*v for k,v in present.items())/sum(v["weight"] for v in RUBRIC123.values()); critical=any(v["critical"] and not present[k] for k,v in RUBRIC123.items()); return score,critical,present
probe123={"conclusion":"A","evidence":[],"limitation":"L"}; score_probe123,critical_probe123,present_probe123=rubric_score123(probe123)
assert math.isclose(score_probe123,.5)
assert critical_probe123 and present_probe123["evidence"]==0
assert set(RUBRIC123)==set(REQUIRED123)

## 2. Critique 是可执行缺陷，不是泛泛评价

每条含 dimension、severity、evidence 和 requested change；禁止 evaluator 直接调用工具或改写答案。parser 拒绝未知维度和无证据批评。高严重度先修，避免一次重写全部导致回归不可归因。

In [ ]:
@dataclass(frozen=True)
class Critique123:
    dimension:str; severity:int; evidence:str; requested_change:str
    def __post_init__(self):
        if self.dimension not in REQUIRED123 or not 1<=self.severity<=3 or not self.evidence or not self.requested_change: raise ValueError("critique_contract")
def evaluate123(draft): return [Critique123(k,3 if k=="evidence" else 1,f"{k} missing",f"add {k}") for k in REQUIRED123 if not draft.get(k)]
critiques123=evaluate123({"conclusion":"A"})
assert {c.dimension for c in critiques123}=={"evidence","limitation"}
assert max(c.severity for c in critiques123)==3
try: Critique123("style",9,"","x"); raise AssertionError("bad critique accepted")
except ValueError as e: assert str(e)=="critique_contract"

## 3. Optimizer 做最小、可追踪修订

修订输入是原草稿与排序后的 critique；每轮只修改目标字段，并生成 diff。真实生成模型可能顺手破坏已有正确内容，所以每轮必须全量重评，不能只检查刚修的维度。

In [ ]:
DEFAULTS123={"conclusion":"采用方案A","evidence":["实验e1"],"limitation":"仅在受控数据验证"}
def optimize123(draft,critiques):
    out=copy.deepcopy(draft)
    if critiques:
        chosen=max(critiques,key=lambda c:(c.severity,-REQUIRED123.index(c.dimension))); out[chosen.dimension]=copy.deepcopy(DEFAULTS123[chosen.dimension]); return out,{"field":chosen.dimension,"before":draft.get(chosen.dimension),"after":out[chosen.dimension]}
    return out,None
d0_123={"conclusion":"采用方案A"}; d1_123,diff1_123=optimize123(d0_123,evaluate123(d0_123))
assert diff1_123["field"]=="evidence" and d1_123["evidence"]==["实验e1"]
assert "limitation" not in d1_123
assert d0_123=={"conclusion":"采用方案A"}

## 4. 有界循环保留 best-so-far

每轮记录 draft hash、score、critical、critique 和成本。达到 pass 且无 critical 结束；新 draft 重复说明 cycle；score 未提高若干轮则 no-progress；预算耗尽返回历史最佳而不是最后版本。

In [ ]:
def draft_hash123(d): return hashlib.sha256(json.dumps(d,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def reflection_loop123(initial,max_rounds=5):
    current=copy.deepcopy(initial); seen=set(); history=[]; best=(float("-inf"),None)
    for round_id in range(max_rounds+1):
        h=draft_hash123(current)
        if h in seen: return best[1],history,"cycle"
        seen.add(h); score,critical,_=rubric_score123(current); history.append({"round":round_id,"score":score,"critical":critical,"hash":h})
        if score>best[0]: best=(score,copy.deepcopy(current))
        if score==1 and not critical: return current,history,"passed"
        if round_id==max_rounds: return best[1],history,"budget"
        current,_=optimize123(current,evaluate123(current))
    raise AssertionError("unreachable")
best123,history123,stop123=reflection_loop123({"conclusion":"采用方案A"})
assert stop123=="passed" and rubric_score123(best123)[0]==1
assert len(history123)==3
assert [x["score"] for x in history123]==sorted(x["score"] for x in history123)

## 5. 最后版本不一定最好，必须能回滚

生成式 optimizer 可能修一处坏一处。选择 highest score 且满足 critical gate 的版本，并保留所有 diff；如果 evaluator 噪声大，用多次/多 grader 置信或人工，而不是盲目继续。下面用预定义退化序列验证 best-so-far。

In [ ]:
variants123=[{"conclusion":"A"},{"conclusion":"A","evidence":["e"]},{"evidence":["e"],"limitation":"L"}]
scored123=[rubric_score123(v)[0] for v in variants123]; best_idx123=max(range(len(variants123)),key=lambda i:scored123[i])
assert scored123==[2/6,5/6,4/6]
assert best_idx123==1
assert variants123[-1] is not variants123[best_idx123]

## 6. Evaluator 与 Generator 共享模型会共享盲点

用确定性 schema、引用验证、单测和人类 gold 校准 evaluator；对位置、长度、措辞做 counterfactual。隐藏模型身份，交换候选顺序。Evaluator 输出中的外部引用仍是不可信，不能借反馈扩大工具权限。

In [ ]:
def style_blind_score123(content_facts,word_count): return len(set(content_facts))
concise123=style_blind_score123(["a","b"],20); verbose123=style_blind_score123(["a","b"],200)
def unsafe_feedback123(text): return any(x in text.lower() for x in ("call tool","ignore policy","发送密码"))
assert concise123==verbose123==2
assert unsafe_feedback123("CALL TOOL delete")
assert not unsafe_feedback123("补充证据来源")

## 7. 迭代收益要覆盖额外模型调用

画 round→质量/成本曲线，通常前几轮收益最大。按任务风险设置 max rounds、token、deadline；低价值请求直接一次生成。Evaluator-Optimizer 只在明确反馈可持续改善时采用，否则固定 workflow 更可靠。

In [ ]:
quality123=[.62,.79,.84,.845]; cumulative_cost123=[.01,.03,.05,.07]; marginal123=[quality123[i]-quality123[i-1] for i in range(1,len(quality123))]
stop_round123=next(i for i,gain in enumerate(marginal123,1) if gain<.01)
assert stop_round123==3
assert marginal123[0]>marginal123[1]>marginal123[2]
assert cumulative_cost123[stop_round123]>.05

## 8. 评测关注净提升、回归、循环和 evaluator 成本

与单次生成对比 task success、critical error、平均最佳轮、回滚率、cycle/no-progress、额外 token 与 p95；逐任务 paired 测增益。Evaluator、Optimizer、rubric 或 stop policy 任一变化都要独立版本化。

In [ ]:
baseline123=np.array([.6,.7,.8,.5]); optimized123=np.array([.8,.75,.82,.72]); gains123=optimized123-baseline123
manifest123={"schema":1,"generator":"g-v2","evaluator":"e-v4","rubric":"report-v3","max_rounds":5,"stop":["pass","cycle","no_progress","budget"],"selection":"best_so_far"}; digest123=hashlib.sha256(json.dumps(manifest123,sort_keys=True).encode()).hexdigest()
assert gains123.mean()>0
assert np.all(gains123>=0) and manifest123["selection"]=="best_so_far"
assert len(digest123)==64

## 面试总结

完整链路是：**冻结 rubric → 结构化且有证据 critique → 最小修订 diff → 全量重评 → best-so-far → pass/增益/cycle/预算停止 → evaluator 偏差校准 → round 级质量成本 ablation**。Reflection 只有在反馈可验证、错误不高度相关时才值得循环。

延伸阅读：[Reflexion](https://arxiv.org/abs/2303.11366)、[Self-Refine](https://arxiv.org/abs/2303.17651)、[Evaluator-Optimizer Pattern](https://www.anthropic.com/engineering/building-effective-agents)。